<a href="https://colab.research.google.com/github/Epot12/QML_project/blob/master/QML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#QML PROJECT

In [2]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00


In [7]:
import os
import shutil
from google.colab import drive
import torch
import torch.nn as nn
import pandas as pd
import pydicom
import glob
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import swin_b, Swin_B_Weights

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Exploratory Data Analysis INbreast

In [9]:
zip_path = '/content/drive/MyDrive/QML/INbreast/INbreast.zip'

# estrazione zip nella memoria temporanea di Colab, NON su Drive
# Questo percorso è veloce e si cancella alla chiusura di Colab
extract_path = '/content/data/INbreast_extracted'

print(f"File ZIP sorgente: {zip_path}")
print(f"Cartella di destinazione: {extract_path}")

# Estrazione (Unzip)
if not os.path.exists(extract_path):
    print("Estrazione in corso...")
    # Creo la cartella
    os.makedirs(extract_path, exist_ok=True)

    # Estraggo lo zip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Estrazione completata!")
else:
    print("Dataset già estratto, procedo.")

# Ora cerco i DICOM nella cartella ESTRATTA (non nello zip)
dataset_root = extract_path  # punta alla cartella aperta!

print(f"Analisi del percorso: {dataset_root}")

dicom_files = []
for root, dirs, files in os.walk(dataset_root):
    for file in files:
        # Controllo euristico e ignoriamo i file di sistema/macOS
        if not file.startswith('.') and not file.endswith(('.xls', '.xlsx', '.csv', '.xml', '.txt', '.zip')):
             dicom_files.append(os.path.join(root, file))

print(f"Trovati {len(dicom_files)} potenziali file DICOM.")

# 4. Estrazione Metadati
data_list = []
print("Lettura metadati DICOM in corso...")

for file_path in dicom_files:
    try:
        ds = pydicom.dcmread(file_path, stop_before_pixels=True)

        # Recupera il valore grezzo dal DICOM
        pid_dicom = ds.get('PatientID', '')

        # Logica di correzione: Se è "Unknown", vuoto o None, usa il nome del file
        filename = os.path.basename(file_path)
        if str(pid_dicom).strip() in ['Unknown', '', 'None']:
            # Pulisce il nome file da estensioni o caratteri extra
            clean_pid = filename.split('_')[0].replace('.dcm', '')
            final_patient_id = clean_pid
        else:
            final_patient_id = pid_dicom

        data_list.append({
            'file_path': file_path,
            'PatientID': final_patient_id,  # Usa il valore corretto calcolato sopra
            'Laterality': ds.get('ImageLaterality', 'Unknown'),
            'ViewPosition': ds.get('ViewPosition', 'Unknown'),
            'Rows': ds.get('Rows', 0),
            'Columns': ds.get('Columns', 0)
        })
    except Exception as e:
        print(f"Errore su {file_path}: {e}")
        continue

df_images = pd.DataFrame(data_list)
print(f"Fatto! Dimensioni dataset: {df_images.shape}")
display(df_images.head())

File ZIP sorgente: /content/drive/MyDrive/QML/INbreast/INbreast.zip
Cartella di destinazione: /content/data/INbreast_extracted
Dataset già estratto, procedo.
Analisi del percorso: /content/data/INbreast_extracted
Trovati 957 potenziali file DICOM.
Lettura metadati DICOM in corso...
Errore su /content/data/INbreast_extracted/INbreast/INbreast Release 1.0/inbreast.pdf: File is missing DICOM File Meta Information header or the 'DICM' prefix is missing from the header. Use force=True to force reading.
Errore su /content/data/INbreast_extracted/INbreast/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/50998440_muscle.roi: File is missing DICOM File Meta Information header or the 'DICM' prefix is missing from the header. Use force=True to force reading.
Errore su /content/data/INbreast_extracted/INbreast/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle ROI/50994589_muscle.roi: File is missing DICOM File Meta Information header or the 'DICM' prefix is missing from the header. Use fo

,file_path,PatientID,Laterality,ViewPosition,Rows,Columns
0,/content/data/INbreast_extracted/INbreast/INbr...,53582540,Unknown,Unknown,3328,2560
1,/content/data/INbreast_extracted/INbreast/INbr...,53581941,Unknown,Unknown,3328,2560
2,/content/data/INbreast_extracted/INbreast/INbr...,51049489,Unknown,Unknown,3328,2560
3,/content/data/INbreast_extracted/INbreast/INbr...,50996999,Unknown,Unknown,4084,3328
4,/content/data/INbreast_extracted/INbreast/INbr...,53581433,Unknown,Unknown,3328,2560


Nel codice seguente si verifica se esistono degli split predefiniti. Inoltre vengono cercate le diagnosi.

In [ ]:
# Cerca il file Excel con le diagnosi (BI-RADS)
xls_files = glob.glob(os.path.join(dataset_root, '**/*.xls*'), recursive=True)
csv_files = glob.glob(os.path.join(dataset_root, '**/*.csv'), recursive=True)
metadata_files = xls_files + csv_files

print("\n--- ANALISI FEATURES E LABELS ---")
if metadata_files:
    print(f"Trovato file di metadati/label: {metadata_files[0]}")
    meta_path = metadata_files[0]

    # Carica a seconda del formato
    if meta_path.endswith('csv'):
        df_labels = pd.read_csv(meta_path, sep=';') # Spesso INbreast usa ; come separatore
    else:
        df_labels = pd.read_excel(meta_path)

    print("Prime righe del file Labels:")
    display(df_labels.head())

    print("\n Colonne disponibili nel file Labels (Features):")
    print(df_labels.columns.tolist())
else:
    print("ATTENZIONE: Nessun file Excel/CSV trovato nella cartella.")
    print("INbreast di solito ha un file 'INbreast.xls'. Senza di esso non ci sono le diagnosi (Benigno/Maligno).")

# Controllo Split (Train/Test)
# INbreast è famoso per NON avere uno split ufficiale. Verifica.
unique_patients = df_images['PatientID'].nunique()
print(f"\n--- STRUTTURA DATASET ---")
print(f"Totale Pazienti unici: {unique_patients}")
print(f"Totale Immagini: {len(df_images)}")

# Check rapido sulla distribuzione delle Viste
if not df_images.empty:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    sns.countplot(data=df_images, x='ViewPosition')
    plt.title('Distribuzione Viste (CC vs MLO)')

    plt.subplot(1, 2, 2)
    sns.countplot(data=df_images, x='Laterality')
    plt.title('Distribuzione Lateralità (L vs R)')

    plt.show()
else:
    print("Nessun dato immagine da visualizzare.")

### Swin B Transformer classico

In [ ]:
weights = Swin_B_Weights.IMAGENET1K_V1
model = swin_b(weights=weights)

In [ ]:
for param in model.parameters():
    param.requires_grad = True

num_features = model.head.in_features

In [ ]:
model.head = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.ReLU(),
    nn.Linear(1024, 16),
    nn.Linear(16, 2)
)